# Fine-tuning of the U-Net on the MDT records of `munich_map_errors`

**Purpose.** This notebook fine-tunes all parameters of the pretrained U-Net on the train
records.

**Inputs.** `lab/outputs/munich_map_errors/models/pretrained*.pt` and the files of
`lab/data/munich_map_errors/processed/`.

**Method.** The predicted RSRP of a received (record, cell) pair is
$P_{\mathrm{RS},c} + \hat{G}_c(g_r)$, where $g_r$ is the grid cell that holds the
reported position of record $r$. The loss is the mean absolute error (MAE) of the RSRP in dB
over the train pairs. A record outside the grid has no pair. The kept parameters have the
smallest MAE on the validation records.

**Outputs.** `lab/outputs/munich_map_errors/models/fine_tuned*.pt` and `lab/outputs/munich_map_errors/histories/`.

**Requirements.** Run `03_pretraining.ipynb` first.

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from experiments import fine_tune_model, history_figure, load_model, load_scenario, save_model
from radio_calibration.evaluation import ResourceMonitor
from utils import load_config, save_figure, save_table, setup_notebook

cfg = load_config("scenario=munich_map_errors")
device = setup_notebook(cfg)
data = load_scenario(cfg)

## 2. Fine-tuning

In [ ]:
model = load_model(data, "pretrained", device)
with ResourceMonitor() as monitor:
    history = fine_tune_model(cfg, data, model)
save_model(data, "fine_tuned", model, history)
histories = {"fine_tuned": history}
resources = {"fine_tuned": monitor.summary()}

In [ ]:
fig = history_figure(histories)
save_figure(fig, "fine_tuning_history")
plt.show()

**Fig. 1.** Fine-tuning. (a) MAE of the train pairs in each epoch. (b) MAE of the
validation pairs.

In [ ]:
save_table(
    pd.DataFrame(
        {
            model_name: {
                "Smallest validation MAE (dB)": history["validation_loss"].min(),
                "Epoch of the smallest validation MAE": history["validation_loss"].idxmin() + 1,
                "Mean time of one step (s)": history["step_seconds"].mean(),
            }
            for model_name, history in histories.items()
        }
    )
    .T.style.set_caption("Table 1. Result of the fine-tuning.")
    .format(precision=4),
    "fine_tuning_result",
)

In [ ]:
save_table(
    pd.DataFrame(resources)
    .T.style.set_caption("Table 2. Resource use of the fine-tuning.")
    .format(precision=1),
    "fine_tuning_resources",
)